# 04 Offline Augmentation

Generate mild offline visual augmentations from the generated training split only.

The PPE detector now has four classes:

- `0 = person`
- `1 = helmet`
- `2 = vest`
- `3 = cleaning_coverall`

Because the current offline transforms do not move objects, labels are copied unchanged and class `3` is preserved automatically.


## Purpose

Notebook 04 reads only:

```text
data/generated/splits/train/images/
data/generated/splits/train/labels/
```

It writes augmented training-only samples to:

```text
data/generated/augmented/images/
data/generated/augmented/labels/
```

Validation and test splits are never read or modified. This prevents augmented siblings from leaking into validation or final test evaluation.


## Setup

Find `v2_pipeline`, import the augmentation helpers, and keep the notebook as orchestration only.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


# Jupyter launch location is not guaranteed. This helper lets every later path
# stay relative to v2_pipeline instead of hardcoding a machine-specific path.
def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline folder from the current working directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate
    raise RuntimeError(
        "Could not find v2_pipeline root. Run this notebook from inside the repository."
    )


V2_ROOT = find_v2_root(Path.cwd()).resolve()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from src.augmentation.offline_blur_compression import (
    generate_blur_compression_augmentation,
)  # noqa: E402
from src.augmentation.offline_ir import generate_ir_augmentation  # noqa: E402
from src.augmentation.offline_sunlight import generate_sunlight_augmentation  # noqa: E402

print(f"v2 root: {V2_ROOT}")

v2 root: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline


## Load Configuration

Dataset paths come from `dataset_config.yaml`; augmentation ratios come from `augmentation_config.yaml`; class names are displayed so the report can be checked against the four-class contract.


In [2]:
config_dir = V2_ROOT / "configs"
with (config_dir / "dataset_config.yaml").open("r", encoding="utf-8") as file_handle:
    dataset_config = yaml.safe_load(file_handle)
with (config_dir / "augmentation_config.yaml").open(
    "r", encoding="utf-8"
) as file_handle:
    augmentation_config = yaml.safe_load(file_handle)
with (config_dir / "class_names.yaml").open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)

splits_dir = V2_ROOT / dataset_config["generated"]["splits"]
augmented_dir = V2_ROOT / dataset_config["generated"]["augmented"]
reports_dir = V2_ROOT / dataset_config["reports"]["augmentation"]
reports_dir.mkdir(parents=True, exist_ok=True)

train_images_dir = splits_dir / "train" / "images"
train_labels_dir = splits_dir / "train" / "labels"
augmented_images_dir = augmented_dir / "images"
augmented_labels_dir = augmented_dir / "labels"

ratios = augmentation_config["offline_augmentation"]
ir_ratio = float(ratios.get("ir_ratio", 0.0))
sunlight_ratio = float(ratios.get("sunlight_ratio", 0.0))
blur_compression_ratio = float(ratios.get("blur_compression_ratio", 0.0))
random_seed = int(dataset_config.get("random_seed", 42))
class_names = {
    int(class_id): str(class_name)
    for class_id, class_name in class_config["names"].items()
}

print(f"train images: {train_images_dir}")
print(f"train labels: {train_labels_dir}")
print(f"augmented images: {augmented_images_dir}")
print(f"augmented labels: {augmented_labels_dir}")
print("Class map:")
for class_id, class_name in class_names.items():
    print(f"- {class_id}: {class_name}")

train images: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\generated\splits\train\images
train labels: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\generated\splits\train\labels
augmented images: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\generated\augmented\images
augmented labels: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\generated\augmented\labels
Class map:
- 0: person
- 1: helmet
- 2: vest
- 3: cleaning_coverall


## Check Training Split

This is the safety gate. If Notebook 03 has not created a training split, augmentation stops before writing anything.


In [3]:
valid_image_extensions = {".jpg", ".jpeg", ".png"}

if not train_images_dir.exists() or not train_labels_dir.exists():
    raise RuntimeError("Training split is missing. Run Notebook 03 before Notebook 04.")

train_image_paths = sorted(
    path
    for path in train_images_dir.iterdir()
    if path.is_file() and path.suffix.lower() in valid_image_extensions
)

if not train_image_paths:
    raise RuntimeError(f"No training images found in {train_images_dir}")

print(f"Original train images found: {len(train_image_paths)}")
print("Validation and test folders are intentionally not inspected in this notebook.")

Original train images found: 596
Validation and test folders are intentionally not inspected in this notebook.


## Intended Augmentation Counts

This preview estimates how many training images each transform will sample. Non-zero ratios select at least one image for very small datasets.


In [4]:
def expected_count(num_images: int, ratio: float) -> int:
    """Return the deterministic sample count used by augmentation helpers."""
    if ratio <= 0:
        return 0
    return min(num_images, max(1, int(round(num_images * ratio))))


intended_counts_df = pd.DataFrame(
    [
        {
            "augmentation_type": "ir",
            "ratio": ir_ratio,
            "expected_selected_images": expected_count(
                len(train_image_paths), ir_ratio
            ),
        },
        {
            "augmentation_type": "sunlight",
            "ratio": sunlight_ratio,
            "expected_selected_images": expected_count(
                len(train_image_paths), sunlight_ratio
            ),
        },
        {
            "augmentation_type": "blur_compression",
            "ratio": blur_compression_ratio,
            "expected_selected_images": expected_count(
                len(train_image_paths), blur_compression_ratio
            ),
        },
    ]
)
display(intended_counts_df)

,augmentation_type,ratio,expected_selected_images
0,ir,0.25,149
1,sunlight,0.15,89
2,blur_compression,0.10,60


## Generate Offline Augmentations

All augmentation types write into the same generated augmented folder using unique filename prefixes (`ir_`, `sun_`, `blur_`). Labels are copied unchanged because these transforms are visual-only and do not move object boxes.


In [5]:
OVERWRITE_EXISTING_AUGMENTED = False

# These functions are class-agnostic: they transform image appearance only. The
# copied labels can contain person, helmet, vest, and cleaning_coverall boxes.
ir_report = generate_ir_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=augmented_images_dir,
    output_labels_dir=augmented_labels_dir,
    ratio=ir_ratio,
    seed=random_seed,
    overwrite=OVERWRITE_EXISTING_AUGMENTED,
)

sunlight_report = generate_sunlight_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=augmented_images_dir,
    output_labels_dir=augmented_labels_dir,
    ratio=sunlight_ratio,
    seed=random_seed,
    overwrite=OVERWRITE_EXISTING_AUGMENTED,
)

blur_report = generate_blur_compression_augmentation(
    images_dir=train_images_dir,
    labels_dir=train_labels_dir,
    output_images_dir=augmented_images_dir,
    output_labels_dir=augmented_labels_dir,
    ratio=blur_compression_ratio,
    seed=random_seed,
    overwrite=OVERWRITE_EXISTING_AUGMENTED,
)

combined_report_df = pd.concat(
    [ir_report, sunlight_report, blur_report],
    ignore_index=True,
)

display(combined_report_df.head(20))

,augmentation_type,original_image_path,original_label_path,augmented_image_path,augmented_label_path,status,notes,num_objects,num_person,num_helmet,num_vest,num_cleaning_coverall
0,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,3,1,1,1,0
1,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,9,4,2,3,0
2,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,9,3,3,3,0
3,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,8,5,1,2,0
4,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,1,1,0,0,0
5,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,17,8,4,5,0
6,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,2,1,0,1,0
7,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,12,4,4,4,0
8,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,9,4,2,3,0
9,ir,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,generated,,8,3,3,2,0


## Save Compact Reports

To keep artifacts manageable, Notebook 04 saves only one detailed combined report and one compact summary. It does not save separate per-augmentation CSVs or example figures by default.


In [6]:
def summarize_augmentation_report(report_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize generated/skipped/failed counts and four-class label counts."""
    if report_df.empty:
        return pd.DataFrame(
            columns=[
                "augmentation_type",
                "selected_images",
                "generated_images",
                "skipped_images",
                "failed_images",
                "num_objects",
                "num_person",
                "num_helmet",
                "num_vest",
                "num_cleaning_coverall",
            ]
        )

    rows = []
    for augmentation_type, group in report_df.groupby("augmentation_type", sort=True):
        generated = group.loc[group["status"].eq("generated")]
        rows.append(
            {
                "augmentation_type": augmentation_type,
                "selected_images": int(len(group)),
                "generated_images": int(group["status"].eq("generated").sum()),
                "skipped_images": int(group["status"].eq("skipped").sum()),
                "failed_images": int(group["status"].eq("failed").sum()),
                # Count only generated rows here so the summary describes labels
                # actually present in data/generated/augmented.
                "num_objects": int(generated["num_objects"].sum()),
                "num_person": int(generated["num_person"].sum()),
                "num_helmet": int(generated["num_helmet"].sum()),
                "num_vest": int(generated["num_vest"].sum()),
                "num_cleaning_coverall": int(generated["num_cleaning_coverall"].sum()),
            }
        )
    return pd.DataFrame(rows)


summary_df = summarize_augmentation_report(combined_report_df)

report_path = reports_dir / "offline_augmentation_report.csv"
summary_path = reports_dir / "offline_augmentation_summary.csv"
combined_report_df.to_csv(report_path, index=False)
summary_df.to_csv(summary_path, index=False)

print(f"Saved combined report: {report_path}")
print(f"Saved summary: {summary_path}")
display(summary_df)

Saved combined report: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\augmentation\offline_augmentation_report.csv
Saved summary: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\augmentation\offline_augmentation_summary.csv


,augmentation_type,selected_images,generated_images,skipped_images,failed_images,num_objects,num_person,num_helmet,num_vest,num_cleaning_coverall
0,blur_compression,60,60,0,0,417,177,116,124,0
1,ir,149,149,0,0,1316,542,383,391,0
2,sunlight,89,89,0,0,804,338,230,236,0


## Final Counts

The generated image count should match the summary unless files were skipped because they already existed or labels were missing.


In [7]:
generated_total = int(combined_report_df["status"].eq("generated").sum())
skipped_total = int(combined_report_df["status"].eq("skipped").sum())
failed_total = int(combined_report_df["status"].eq("failed").sum())

final_counts_df = pd.DataFrame(
    [
        {"item": "original train images", "count": len(train_image_paths)},
        {"item": "augmented images generated", "count": generated_total},
        {"item": "selected images skipped", "count": skipped_total},
        {"item": "selected images failed", "count": failed_total},
        {
            "item": "generated cleaning_coverall objects",
            "count": int(summary_df["num_cleaning_coverall"].sum())
            if not summary_df.empty
            else 0,
        },
    ]
)
display(final_counts_df)

,item,count
0,original train images,596
1,augmented images generated,298
2,selected images skipped,0
3,selected images failed,0
4,generated cleaning_coverall objects,0


## Final Checklist Before Notebook 05

- Augmentation read only `data/generated/splits/train`.
- Validation and test splits were not read or modified.
- Augmented files were written under `data/generated/augmented/images` and `labels`.
- Labels were copied unchanged, preserving `person`, `helmet`, `vest`, and `cleaning_coverall` boxes.
- Only `offline_augmentation_report.csv` and `offline_augmentation_summary.csv` were saved.
- No model training or ablation dataset building was run.
